In [ ]:
%run 0_requirements.ipynb

In [ ]:
meses = pd.date_range(
    end=pd.Timestamp.today().replace(day=1),
    periods=4,
    freq="MS"
)

dfs = []

for data in meses:  # meses criado com pd.date_range(...)
    ano = data.year
    mes = f"{data.month:02d}"

    print(f"\nConsultando {mes}/{ano}")

    urls = [
        f"https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan/{ano}/{mes}-dados-abertos-precos-{ano}-{mes}-diesel-gnv.csv",
        f"https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan/{ano}/{mes}-diesel-gnv.csv",
        f"https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan/{ano}/precos-diesel-gnv-{mes}.csv",
        f"https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan/{ano}/{mes}-dados-abertos-precos-diesel-gnv.csv",
        f"https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc/dsan/{ano}/{mes}-dados-abertos-precos-diesel-gnv"
    ]

    for url in urls:
        try:
            df_mes = pd.read_csv(url, sep=";")  # remova sep=";" se o arquivo usar vírgula
            print(f"Quantidade de dados: {len(df_mes)}")
            print(f"Arquivo encontrado: {url}")
            dfs.append(df_mes)  # guarda o DataFrame
            break  # achou um arquivo; passa para o próximo mês
        except HTTPError as erro:
            if erro.code != 404:
                print(f"Nenhum dos links foi encontrado para {mes}/{ano}")
                # raise
    else:
        # raise FileNotFoundError(f"Nenhum dos links foi encontrado para {mes}/{ano}")
        print(f"Nenhum dos links foi encontrado para {mes}/{ano}")


df_anp = pd.concat(dfs, ignore_index=True)
df_anp = df_anp[df_anp["Produto"] == "DIESEL S10"]

In [ ]:
print(df_anp.info())
print(df_anp.describe())
df_anp.head()

In [ ]:
df_anp.nunique()

In [ ]:
df_anp["Data da Coleta"] = pd.to_datetime(
    df_anp["Data da Coleta"],
    dayfirst=True,
    errors="coerce"
)

df_anp["Valor de Venda"] = pd.to_numeric(
    df_anp["Valor de Venda"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)

df_anp["anomes"] = df_anp["Data da Coleta"].dt.strftime("%Y-%m")
df_anp.reset_index()
df_anp.info()

In [ ]:
df_anp = df_anp[["anomes", "Regiao - Sigla", "Estado - Sigla", "Municipio", "Produto",
                  "Data da Coleta", "Valor de Venda", "Bandeira"]]

df_anp.info()

In [ ]:
visao = "estado"
local = "SP"

In [ ]:
frete_o = Frete()
df_anp_hist, resultado = frete_o.calcular(df_anp = df_anp, eixos=3, distancia_km=100)

print(resultado["custos_por_km"])
print(f"Preco medio do combustivel: R$ {resultado["preco_combustivel"]:.2f}")
print(f"Custo médio por km: R$ {resultado['custo_medio_por_km']:.2f}")
print(f"Preco médio por km: R$ {resultado['preco_medio_por_km']:.2f}")
print(f"Margem média por km: R$ {resultado['margem_media_por_km']:.2f}")
print(f"Custo estimado: R$ {resultado['custo_estimado']:.2f}")
print(f"Frete estimado: R$ {resultado['frete_estimado']:.2f}")
print(f"Margem estimado: R$ {resultado['margem_estimada']:.2f}")

In [ ]:
hist, s10 = resumo_diesel_s10(df_anp)
hist